# IgniteCyber 5‑Day Cyber AI Analyst Bootcamp — LAB1.1

**Offline-only** • **Sigma-first** • **Elastic/Kibana + TheHive 5 + Cortex + MISP**  
**Datasets:** `/opt/bootcamp/datasets` • **Notebooks:** `/opt/bootcamp/notebooks`

## CJ Intel Spine (Single evolving MISP Event + one Event Report)
Throughout the week, we maintain **one** MISP event for the threat actor **Cinder Jackal (CJ)** and append a **single MISP Event Report** day-by-day.

**Workflow (Notebook-guided / manual in TheHive UI):**
1. Hunt/confirm in **Kibana** (`zeek-*`, `wazuh-archives-*`, `elastic-agent-*`)
2. Add observables + evidence to **TheHive case**
3. Run **Cortex analyzers** from TheHive (sightings / file triage)
4. Promote validated intel into **MISP objects**
5. Append the narrative block generated by this notebook into the **CJ MISP Event Report**

## Required dataset bundles for this lab
- (see course dataset catalog for this lab bundle)


# Lab 1.1 — Hosted Lab Onboarding + Health Checks


## Scenario storyline (persistent across the week)

**Company:** ApexFin Solutions (AF) — a fast-growing fintech startup that provides a consumer budgeting app and small-business payment rails.

**Business critical assets:**
- AF-PAY API (customer payments)
- AF-PORTAL (employee/admin portal)
- AF-ANALYTICS (data warehouse exports)

**Threat actor:** CINDER JACKAL (fictional) — financially motivated intrusion set that blends commodity tradecraft with “living-off-the-land” execution and quick monetization.

**CINDER JACKAL objectives:**
1) Obtain employee credentials via phishing and browser session theft  
2) Establish persistence and stealthy C2  
3) Identify and exfiltrate customer PII and payment reconciliation data  
4) Extort FinanceFront with data-theft + disruption pressure

We use **MITRE ATT&CK** to describe behavior, **Sigma-first detections** for portability, and **Elastic/Kibana + Wazuh/Zeek** telemetry for investigations.



### Lab VM / Data assumptions (offline-only)

- Datasets (ZIP) are available locally in: `/opt/bootcamp/datasets`
- Elastic/Kibana index patterns:
  - `zeek-*`
  - `wazuh-archives-*`
  - `elastic-agent-*`
- Elastic user/pass are preconfigured in the VM (read from environment variables by default).
- Local LLM: **Ollama** on `http://localhost:11434` (no internet required).


## Objectives
- Complete the tasks for **Lab 1.1** and capture evidence.
- Map observed behavior to MITRE ATT&CK.
- Produce a Sigma-first detection outcome.

## MITRE ATT&CK mapping
- **TA0007 / T1082** — System Information Discovery

## Datasets (offline ZIPs)
_(No external dataset required — use the pre-indexed lab telemetry.)_

## Tasks (do these in order)
1. Verify Elastic cluster health and authenticate.
1. Confirm that index patterns (`zeek-*`, `wazuh-archives-*`, `elastic-agent-*`) return data.
1. Record ingestion status (healthy/yellow/red), unassigned shards, and most recent event timestamp per index family.


> Attribution / inspiration: see `docs/references.md` (Security Break Jupyter Collection + Security Datasets).


In [ ]:
import os
from bootcamp_lib import *
print('ES_HOST:', ES_HOST)
print('Ollama models:', ollama_models()[:5])
print('Dataset ZIP count:', len(list_dataset_zips()))


## Step 1 — Elastic health checks


In [ ]:
info = es_get('/')
print(json.dumps({k: info.get(k) for k in ['name','cluster_name','cluster_uuid','version','tagline']}, indent=2))
health = es_get('/_cluster/health')
print(json.dumps({k: health.get(k) for k in ['status','number_of_nodes','active_primary_shards','active_shards','unassigned_shards']}, indent=2))


## Step 2 — Confirm index patterns exist (zeek, wazuh, elastic-agent)


In [ ]:
aliases = es_get('/_aliases')
idx = sorted(list(aliases.keys()))
patterns = ['zeek-','wazuh-alerts-','elastic-agent-']
found = {p:[i for i in idx if i.startswith(p)] for p in patterns}
for p, items in found.items():
    print(p, '=>', len(items), 'indices')
    for i in items[:10]:
        print('  -', i)


## Step 3 — Measure data coverage and anchor the lab to absolute time

For onboarding, do **not** assume `last 24 hours`. First confirm what time range the dataset actually covers.

### What to record
- Document count per index family
- Earliest timestamp seen
- Latest timestamp seen
- Which source families are usable in this VM right now

> Goal: produce a trustworthy **absolute time window** for Kibana and the rest of the lab notebooks.


In [ ]:
import pandas as pd

index_families = ['zeek-*', 'wazuh-archives-*', 'elastic-agent-*']
coverage_rows = []

for family in index_families:
    row = {"pattern": family, "doc_count": 0, "earliest_ts": None, "latest_ts": None, "status": "missing"}
    try:
        count_info = es_get(f'/{family}/_count')
        row["doc_count"] = count_info.get("count", 0)

        newest = es_get(f'/{family}/_search?size=1&sort=@timestamp:desc')
        oldest = es_get(f'/{family}/_search?size=1&sort=@timestamp:asc')

        newest_hit = ((newest.get("hits") or {}).get("hits") or [{}])[0]
        oldest_hit = ((oldest.get("hits") or {}).get("hits") or [{}])[0]

        row["latest_ts"] = ((newest_hit.get("_source") or {}).get("@timestamp"))
        row["earliest_ts"] = ((oldest_hit.get("_source") or {}).get("@timestamp"))

        if row["doc_count"] > 0:
            row["status"] = "present"
        else:
            row["status"] = "no hits"
    except Exception as e:
        row["status"] = f"error: {e}"

    coverage_rows.append(row)

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

valid_starts = [x for x in coverage_df["earliest_ts"].tolist() if x]
valid_ends = [x for x in coverage_df["latest_ts"].tolist() if x]

lab_time_window = {
    "start": min(valid_starts) if valid_starts else None,
    "end": max(valid_ends) if valid_ends else None,
}

print("Recommended absolute Kibana time window:")
print(json.dumps(lab_time_window, indent=2))


## Step 4 — Build the lab source contract

Use the coverage findings to decide which telemetry families are authoritative for this VM.

### Decision rules
- Prefer `zeek-*` for **network** validation
- Prefer `elastic-agent-*` for **endpoint** validation when it exists
- Fall back to `wazuh-archives-*` for **endpoint / alert-centric** validation if `elastic-agent-*` is missing
- Mark any absent family as a **lab limitation**, not as a failure by itself

> Goal: document what this lab can honestly validate before moving into later investigations.


In [ ]:
present = set(coverage_df.loc[coverage_df["doc_count"].fillna(0) > 0, "pattern"].tolist())

source_contract = {
    "time_mode": "absolute",
    "baseline_mode": "environment-readiness",
    "network_source": "zeek-*" if "zeek-*" in present else "none",
    "endpoint_source": (
        "elastic-agent-*"
        if "elastic-agent-*" in present
        else ("wazuh-archives-*" if "wazuh-archives-*" in present else "none")
    ),
    "case_intel_tools": ["TheHive", "Cortex", "MISP"],
    "limitations": [],
}

if "elastic-agent-*" not in present and "wazuh-archives-*" in present:
    source_contract["limitations"].append(
        "elastic-agent-* is absent; endpoint visibility is alert-centric through wazuh-archives-*."
    )
if "zeek-*" not in present:
    source_contract["limitations"].append(
        "zeek-* is absent; network validation is limited in this VM."
    )
if not lab_time_window.get("start") or not lab_time_window.get("end"):
    source_contract["limitations"].append(
        "Could not derive full time bounds from sample results; verify in Kibana."
    )

print(json.dumps(source_contract, indent=2))


## Step 5 — Produce a readiness checklist and starter queries

Before collecting evidence, write down whether the lab is ready and which Kibana queries you will use.

### Minimum checklist
- Cluster status and shard health recorded
- Active index families identified
- Absolute time window selected
- Network source selected
- Endpoint source selected
- Known limitations noted

> Goal: leave this step with a clean handoff into investigation work and later labs.


In [ ]:
active_patterns = [p for p in index_families if p in present]
target_indices = ",".join(active_patterns)

time_filter = (
    f'@timestamp:["{lab_time_window["start"]}" TO "{lab_time_window["end"]}"]'
    if lab_time_window.get("start") and lab_time_window.get("end")
    else "*"
)

starter_queries = {
    "all_available_sources": time_filter,
    "network_validation": time_filter if source_contract["network_source"] != "none" else "(network source unavailable)",
    "endpoint_validation": time_filter if source_contract["endpoint_source"] != "none" else "(endpoint source unavailable)",
}

readiness = {
    "cluster_status": health.get("status") if "health" in globals() else "unknown",
    "active_primary_shards": health.get("active_primary_shards") if "health" in globals() else None,
    "unassigned_shards": health.get("unassigned_shards") if "health" in globals() else None,
    "active_patterns": active_patterns,
    "target_indices": target_indices,
    "absolute_time_window": lab_time_window,
    "source_contract": source_contract,
}

print("Lab readiness summary:")
print(json.dumps(readiness, indent=2))

print("\nStarter query strings:")
for name, query in starter_queries.items():
    print(f"- {name}: {query}")


## Step 6 — Starter investigation queries (copy/paste into Kibana)

Use the **absolute** time window discovered in Step 3, not a rolling `last 24 hours` filter.

### Suggested starting point
- Start with the `starter_queries` printed in Step 5
- Validate that each active source family actually returns documents
- Note any index family that is present but operationally incomplete for this VM

### If you want manual Kibana checks
- Use the **absolute time picker** from `lab_time_window`
- Query `zeek-*` if network data is present
- Query the selected endpoint source from `source_contract`


## Step 7 — Build an evidence packet (for reporting + AI assistance)

Use a **readiness-focused** query first. For this lab, the evidence packet should prove:
- which index families are present
- what the dataset time bounds are
- whether the VM is ready for later investigations

Adjust `sample_qs` only after you confirm the source contract in Steps 3–6.


In [ ]:
sample_qs = starter_queries.get("all_available_sources", "*") if "starter_queries" in globals() else "*"
target = target_indices if "target_indices" in globals() and target_indices else 'elastic-agent-*,wazuh-archives-*,zeek-*'

# Refine this query only after you verify coverage, time bounds, and source availability.
res = es_qs(target, sample_qs, size=300)
df = hits_to_df(res)
out_path = save_evidence_csv(df, 'Lab_1.1')
print('Saved evidence CSV:', out_path, 'rows:', len(df))
display(df.head(15))


## Step 8 — AI assistance (local Ollama) + human verification
**AI task for this lab:** Have Ollama explain what each index family represents and output a 5-bullet 'lab readiness' checklist.

Rule: **No evidence → no claims.** Validate output with additional queries.


In [ ]:
models = ollama_models()
model = models[0] if models else "llama3"
evidence_preview = df.head(30).to_csv(index=False) if 'df' in globals() and isinstance(df, pd.DataFrame) else ""
prompt = f'''You are a SOC threat hunter. Work offline. Use only the evidence provided.

OUTPUT FORMAT (strict):
- Summary (5 bullets)
- ATT&CK techniques (ID + 1-line justification referencing evidence columns)
- Validation queries (3 KQL queries)
- Containment actions (2)
- Uncertainty (what is missing / what could be wrong)

EVIDENCE (CSV excerpt):
{evidence_preview}
'''
if evidence_preview.strip():
    response = ollama_generate(prompt, model=model, temperature=0.2)
    print(response)
else:
    print("No evidence dataframe found yet. Run Step 7 first.")


## Deliverables / Checkpoint
- Evidence CSV path (saved in `/tmp/`)
- ATT&CK technique IDs you verified
- Sigma rule or tuning notes (if applicable)
- 1-paragraph narrative (what happened + why you believe it)


## CJ Event Report (copy/paste into MISP → Event → Event Reports)

**Report name (single for the week):** `Cinder Jackal — Weekly Narrative (Bootcamp)`  
In MISP: open the **CJ Event** → **Event Reports** → open the weekly report → paste the block below at the end.

> Tip: Keep headings consistent so the report reads like a timeline.

### Paste this block


In [ ]:
from datetime import datetime
import os, textwrap

LAB_CODE = "LAB1.1"
DAY_LABEL = os.getenv("BOOTCAMP_DAY", "") or ""
TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%SZ")

# Fill these as you work the lab (concise + evidence-driven)
findings = {
    "summary": "",
    "what_happened": [],
    "key_iocs": [],
    "attack_mapping": [],
    "elastic_evidence": [],
    "thehive_actions": [],
    "misp_updates": [],
    "next_steps": [],
}

def _lines(items, fallback="(fill in)"):
    return "\n".join(["- " + x for x in (items if items else [fallback])])

def render_report_block():
    md = f"""\
## {DAY_LABEL + " — " if DAY_LABEL else ""}{LAB_CODE} — CJ Narrative Update
**Timestamp (UTC):** {TS}

**Summary:** {findings["summary"] or "(fill in)"}

**What happened**
{_lines(findings["what_happened"])}

**Key IOCs**
{_lines(findings["key_iocs"])}

**MITRE ATT&CK mapping**
{_lines(findings["attack_mapping"])}

**Elastic evidence (queries / fields / indices)**
{_lines(findings["elastic_evidence"])}

**TheHive actions**
{_lines(findings["thehive_actions"])}

**MISP updates (objects + tags)**
{_lines(findings["misp_updates"])}

**Next steps**
{_lines(findings["next_steps"])}

---"""
    return textwrap.dedent(md)

report_block = render_report_block()
print(report_block)

out_dir = "/opt/bootcamp/reports/cj"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, f"{LAB_CODE.replace('.','_')}_cj_event_report_update.md")
with open(out_path, "w", encoding="utf-8") as f:
    f.write(report_block)

print(f"Saved: {out_path}")


## Daily Executive Summary Checkpoint

Use this checkpoint at the end of each lab day to build the rolling executive summary for the Cinder Jackal investigation. The AI assistant can help turn validated evidence into executive-ready language, but it must not introduce new facts.

**Required workflow:**
1. Confirm the lab evidence packet or report block is filled in.
2. Ask the local AI assistant to draft a concise executive update from that evidence only.
3. Validate every claim against notebook output, Elastic/OpenSearch results, TheHive tasks, MISP updates, and source artifacts.
4. Save the final daily checkpoint under `/opt/bootcamp/reports/cj/` and append approved content to the weekly CJ Event Report.


In [ ]:
# Daily Executive Summary Checkpoint - AI-assisted, evidence-first
from datetime import datetime
import os, textwrap, json

_checkpoint_lab = globals().get("LAB_CODE", "LAB-UNKNOWN")
_checkpoint_day = globals().get("DAY_LABEL", "") or os.getenv("BOOTCAMP_DAY", "") or "Day checkpoint"
_checkpoint_ts = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%SZ")

# Prefer the validated CJ Event Report block when the notebook has one.
_checkpoint_evidence = ""
if "report_block" in globals() and str(report_block).strip():
    _checkpoint_evidence = str(report_block)
elif "findings" in globals():
    _checkpoint_evidence = json.dumps(findings, indent=2)
elif "report" in globals():
    _checkpoint_evidence = json.dumps(report, indent=2)
else:
    _checkpoint_evidence = "No structured report block found yet. Fill in the lab findings before finalizing this checkpoint."

_daily_exec_prompt = f"""
You are assisting a SOC analyst with an executive update for the IgniteCyber bootcamp.
Use only the validated evidence below. Do not invent facts, impacts, victims, or containment actions.
If evidence is missing, say what must be validated before leadership reporting.

Return this exact structure:
1. Executive summary: 3-5 bullets, business-focused, no jargon unless necessary.
2. What changed today: short paragraph.
3. Current risk: Low/Medium/High with evidence-based justification.
4. Validated evidence: 3-6 bullets with field names, artifacts, or source systems.
5. Decisions or actions needed: 2-4 bullets.
6. Open questions: 2-4 bullets.
7. Analyst validation checklist: claims that must be checked before sending.

Lab: {_checkpoint_lab}
Day: {_checkpoint_day}
Timestamp UTC: {_checkpoint_ts}

VALIDATED EVIDENCE PACKET:
{_checkpoint_evidence}
""".strip()

_daily_template = f"""# {_checkpoint_day} - Executive Summary Checkpoint

**Lab:** {_checkpoint_lab}  
**Timestamp (UTC):** {_checkpoint_ts}  
**Status:** Draft - requires analyst validation

## Executive Summary
- (AI-assisted draft or analyst-written summary goes here.)

## What Changed Today
- (Summarize the investigation progress from this lab.)

## Current Risk
- (Low/Medium/High and why.)

## Validated Evidence
- (Cite source systems, fields, observables, or artifacts.)

## Decisions or Actions Needed
- (List leadership or SOC actions.)

## Open Questions
- (List unknowns and validation gaps.)

## Analyst Validation Checklist
- [ ] Claims match source evidence.
- [ ] No AI-only claims remain.
- [ ] IOCs are validated before MISP promotion.
- [ ] TheHive tasks reflect current status.
- [ ] Executive wording is accurate and appropriately scoped.
"""

print(_daily_template)
print("\n--- AI prompt prepared for local assistant ---\n")

_models = ollama_models() if "ollama_models" in globals() else []
if _models and "ollama_stream" in globals():
    print(f"Using Ollama model: {_models[0]}\n")
    ollama_stream(_daily_exec_prompt, model=_models[0], temperature=0.2)
elif _models and "ollama_generate" in globals():
    print(f"Using Ollama model: {_models[0]}\n")
    print(ollama_generate(_daily_exec_prompt, model=_models[0], temperature=0.2))
else:
    print("No local Ollama helper/model is available in this notebook session.")
    print("Use the template above, or paste _daily_exec_prompt into the approved local AI workflow.")

_out_dir = "/opt/bootcamp/reports/cj"
os.makedirs(_out_dir, exist_ok=True)
_safe_lab = _checkpoint_lab.replace(".", "_").replace(" ", "_").replace("-", "_")
_out_path = os.path.join(_out_dir, f"{_safe_lab}_daily_executive_summary_checkpoint.md")
with open(_out_path, "w", encoding="utf-8") as f:
    f.write(_daily_template)

print(f"\nSaved daily executive summary checkpoint template: {_out_path}")
